# Test AP by number size for the saved checkpoints

Re-scores the checkpoints of the earlier runs on the test split, adding AP for small
(< 8 px), medium (8-16 px) and large (> 16 px) numbers. Nothing is retrained; it takes a few
minutes.

**Kaggle settings:** Accelerator *GPU T4 x2*, Internet *on*. Under *Input → Add Input → Your
Work*, attach the `jersey-2` dataset and the output of these notebooks:
`jerseynet-training-jersey-2`, `jerseynet-ablation-jersey-2`, `yolov8n-baselines-jersey-2` and
`yolov8n-scratch-384px-jersey-2`. Then *Save Version → Save & Run All (Commit)*. Results go to
`/kaggle/working/rescore_results`. The overall AP50:95 in the table should match the earlier
results; that check comes first.


In [ ]:
# Settings
import os

REPO_URL = "https://github.com/MHKhan50/jersey-number-detection.git"
REPO_REF = "main"
OUT_DIR = "/kaggle/working/rescore_results"
INPUT_ROOTS = ["/kaggle/input"]
print({"REPO_REF": REPO_REF, "OUT_DIR": OUT_DIR})


In [ ]:
# Environment check and Ultralytics (PyTorch from the Kaggle image is kept as it is)
import subprocess
import sys

import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Set Accelerator to GPU T4 x2.")
try:
    import ultralytics  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3"], check=True)
import ultralytics

print("ultralytics", ultralytics.__version__)

In [ ]:
# Get the code
import time
from pathlib import Path

REPO_DIR = Path(os.environ.get("JND_REPO_DIR", "/tmp/jersey-number-detection"))
if not (REPO_DIR / "src" / "jersey_detect").exists():
    for attempt in range(3):
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)]).returncode == 0:
            break
        time.sleep(10)
SRC = str(REPO_DIR / "src")
sys.path.insert(0, SRC)
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Find the dataset
from jersey_detect.prepare import find_dataset

DATA_ROOT = find_dataset(INPUT_ROOTS)
if DATA_ROOT is None:
    raise RuntimeError("No dataset: attach jersey-2 under Input (folder with train/, valid/, test/)")
print("Dataset:", DATA_ROOT)


In [ ]:
# Re-score every checkpoint found under the inputs
env = dict(os.environ, PYTHONPATH=SRC, PYTHONUNBUFFERED="1")
cmd = [sys.executable, "-m", "jersey_detect", "rescore", "--data", str(DATA_ROOT),
       "--checkpoints", *INPUT_ROOTS, "--out", OUT_DIR]
print(" ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
print("return code:", proc.wait())
